# DalMax — Google Colab Pro runbook (notebook form)

Runnable companion to [`COLAB_RUNBOOK.md`](https://github.com/MarioCarvalhoBr/dalmax-framework-deep-active-learning-python/blob/main/COLAB_RUNBOOK.md) — every cell below
mirrors a numbered section of that runbook. **`COLAB_RUNBOOK.md` is the
source of truth**; if this notebook and the runbook ever disagree, trust the
runbook (and re-sync this notebook).

**Never run this notebook on the local no-GPU dev notebook**
(`.claude/rules/data-safety.md`) — it is for Google Colab Pro only.

Run every cell top to bottom, in order. The **Parameters** cell right below
is the only one you should normally need to edit.

In [ ]:
import os

# Parameters -- the ONLY cell in this notebook you should need to edit.
DRIVE_ROOT = "/content/drive/MyDrive/UFMS/Pós-graduação/Doutorado/FINAL/PROJETO/DALMAX"  # per account: the folder holding DATA/ on THAT Drive
REPO_URL = "https://github.com/MarioCarvalhoBr/dalmax-framework-deep-active-learning-python.git"
REPO_DIR = "/content/dalmax"
BRANCH = "main"  # must contain the 2026-11-11 fix (ADR 0014); use "fix/degenerate-baselines" until it is merged into main
PART = "paper1_rerun_a"  # which part(s) of the selected manifest the run cell executes. Default (2026-11-11, ADR 0014): the re-run of the
                 # seven paper-1 baselines fixed on 2026-11-11, split over TWO Colab accounts running at the same time:
                 #   account A keeps PART = "paper1_rerun_a" (24 runs: AdversarialBIM nq100/nq50, KMeansSampling, KCenterGreedy);
                 #   account B sets  PART = "paper1_rerun_b" (39 runs: AdversarialBIM nq10 + the four MC-dropout strategies).
                 # Each account archives the superseded runs first (section 6d cells below). A re-run resumes where it stopped.
                 # The KMH sweep (ADR 0012, 165 runs) is complete: to revisit it set MANIFEST to the sweep manifest and PART = "all".
                 # Main campaign parts: all | paper1 | paper1_rerun_a | paper1_rerun_b | upper_bound | rnhal | texhal | texhal_q17 | hier_transfer
                 # (comma-separated allowed, e.g. "rnhal,texhal").
                 # Session order used for the first execution (job counts from `python -m dalmax.campaign list`):
                 #   1) "paper1"      117 runs (since ADR 0014: 54, plus paper1_rerun_a 24 and paper1_rerun_b 39)
                 #   2) "rnhal"        42 runs
                 #   3) "texhal"       30 runs
                 #   3b) "texhal_q17"  18 runs (TexHAL Q=17 re-run of 6.2/6.3, ADR 0011)
                 #   3c) "hier_transfer" 12 runs (ADR 0013, MANIFEST = ""; see section 6c)
                 #   4) "upper_bound"   3 runs
                 #   (all = 222 runs total)
                 # For the KMH sweep (ADR 0012) set MANIFEST below and PART = "kmh_nq100" | "kmh_nq50" | "kmh_nq10"
                 # (recommended order: kmh_nq100 first, then kmh_nq50, then kmh_nq10).
                 # Re-running the same PART resumes: completed runs are skipped, so a
                 # disconnect loses at most the run in progress.

MANIFEST = ""  # "" = main campaign (manifest.json), the current job (paper1_rerun_a / paper1_rerun_b); the KMH sweep (ADR 0012, files_config/campaign/manifest_kmh_sweep.json) is complete;
               # set MANIFEST = "" for the main campaign (executed 2026-09-30 to 2026-11-01; its runs are skipped as complete).
MANIFEST_ARG = f"MANIFEST={MANIFEST}" if MANIFEST else ""
# Results root / report dirs / Drive copy name depend on the manifest.
if "kmh_sweep" in MANIFEST:
    RESULTS_ROOT, REPORT_ARGS, REPORT_DIR, DRIVE_TABLES = "results/campaign_kmh", "REPORT_ROOT=results/campaign_kmh REPORT_OUT=docs/results/campaign_kmh", "docs/results/campaign_kmh", "campaign_kmh_tables"
else:
    RESULTS_ROOT, REPORT_ARGS, REPORT_DIR, DRIVE_TABLES = "results/campaign", "", "docs/results/campaign", "campaign_tables"

# Also export as env vars so every `!`-shell cell below (and
# scripts/colab/setup_colab.sh's own DRIVE_ROOT override) sees them too.
os.environ["DRIVE_ROOT"] = DRIVE_ROOT
os.environ["PART"] = PART
os.environ["DRIVE_TABLES"] = DRIVE_TABLES
os.environ["REPORT_DIR"] = REPORT_DIR

print("DRIVE_ROOT:", DRIVE_ROOT)
print("REPO_URL:  ", REPO_URL)
print("REPO_DIR:  ", REPO_DIR)
print("BRANCH:    ", BRANCH)
print("PART:      ", PART)
print("MANIFEST:  ", MANIFEST or "(main campaign manifest)")

# Expected size of the two re-run parts (ADR 0014); the gate cell right before the run cell asserts it.
RERUN_EXPECTED = {"paper1_rerun_a": (24, 8), "paper1_rerun_b": (39, 13)}
EXPECTED_RUNS, EXPECTED_GROUPS = RERUN_EXPECTED.get(PART, (None, None)) if MANIFEST == "" else (None, None)
print()
print("#" * 78)
if EXPECTED_RUNS is not None:
    print(f"#  SELECTED PART: {PART}  ->  expect {EXPECTED_RUNS} runs ({EXPECTED_GROUPS} run groups)")
    print("#  Account A runs paper1_rerun_a (24 runs, 8 groups); account B runs paper1_rerun_b (39 runs, 13 groups).")
    print("#  Wrong account/part? Stop now, edit PART above and re-run this cell.")
else:
    print(f"#  SELECTED PART: {PART}  (not a re-run part; no expected count is enforced)")
print("#" * 78)


## 0. Runtime check

See `COLAB_RUNBOOK.md` section 0. Select a GPU runtime (any); the cell below lists whichever GPU Colab assigned this session.
Its name is recorded automatically in every run's `run_metadata.json` (`environment.gpus`).

In [ ]:
!nvidia-smi

In [ ]:
import sys

# Informational only: Colab's SYSTEM Python is 3.13 (as of 2026-09-29), which is
# expected and fine. The project runs in a Poetry venv built on a uv-managed
# Python 3.12 (section 3); that venv interpreter is validated right after install.
print(f"Kernel (system) Python: {sys.version.split()[0]} -- any version is OK here")

## 1. Mount Google Drive

See `COLAB_RUNBOOK.md` section 1.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
assert os.path.isdir(f"{DRIVE_ROOT}/DATA/daninhas_full"), (
    f"{DRIVE_ROOT}/DATA/daninhas_full not found. Check DRIVE_ROOT in the "
    "Parameters cell above, and that daninhas_full was uploaded to Drive at "
    "that path -- every later step assumes it (COLAB_RUNBOOK.md section 1)."
)
print("Dataset folder found on Drive.")

In [ ]:
!ls "$DRIVE_ROOT/DATA/daninhas_full" | head

## 2. Clone or pull the repo

Idempotent: clones fresh into `/content/dalmax` (local runtime disk, not
Drive -- see `COLAB_RUNBOOK.md`'s hybrid-layout decision) the first time, or
pulls `BRANCH` if this runtime already has it (rare -- Colab runtimes are
usually fresh each session, but harmless to check). See
`COLAB_RUNBOOK.md` section 2.

In [ ]:
import subprocess

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"{REPO_DIR} already exists -- pulling latest {BRANCH}...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
else:
    print(f"Cloning {REPO_URL} (branch {BRANCH}) into {REPO_DIR}...")
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True
    )

# os.chdir affects `!`-cells too (they share the process cwd), but `%cd` is
# the documented/safe way to also update Colab's own displayed cwd state.
os.chdir(REPO_DIR)
%cd $REPO_DIR
!git log --oneline -1

## 3. Install dependencies with Poetry

See `COLAB_RUNBOOK.md` section 3.

In [ ]:
# Colab's system Python is 3.13, which torch==2.5.0 has no wheels for.
# Install a separate Python 3.12 with uv and point Poetry at it (idempotent).
!pip install -q uv poetry
!uv python install 3.12

In [ ]:
# Run from the repo root (REPO_DIR) so Poetry picks up pyproject.toml.
%cd {REPO_DIR}
!poetry env use "$(uv python find 3.12)"
!poetry install

This is a fresh runtime each session, so `poetry install` re-downloads the
pinned `torch==2.5.0`/`torchvision==0.20.0` CUDA wheels every time -- expect
roughly **5 minutes** (not previously measured on Colab specifically; treat
as an estimate). It creates an in-project `.venv/` (built on the uv-managed Python 3.12) on the local runtime disk,
same as the lab machine and local notebook.

**Alternative (not recommended by default)**:
`poetry config virtualenvs.create false --local` installs into the runtime's
system Python instead -- slightly faster, but downgrades Colab's preinstalled
torch build to the pinned 2.5.0 and loses environment isolation. See
`COLAB_RUNBOOK.md` section 3 for the full trade-off.

The SYSTEM Python on Colab is 3.13 (expected); it is only used to run `uv`/`poetry`.
The check cell below validates the venv interpreter instead.

In [ ]:
import subprocess

# Validate the Poetry venv interpreter (not the kernel's own Python).
probe = subprocess.run(
    ["poetry", "run", "python", "-c",
     "import sys, torch; print(sys.version.split()[0], torch.__version__, torch.cuda.is_available())"],
    capture_output=True, text=True, cwd=REPO_DIR,
)
print(probe.stdout.strip(), probe.stderr.strip()[-500:])
venv_version = probe.stdout.split()[0] if probe.returncode == 0 and probe.stdout.split() else ""
if not venv_version.startswith("3.12"):
    raise RuntimeError(
        f"Poetry venv Python is {venv_version or 'unavailable'}, expected 3.12.x "
        "(supported range 3.10-3.12; torch==2.5.0 has no 3.13 wheels). "
        "Re-run the previous cell: `uv python install 3.12`, then "
        '`poetry env use "$(uv python find 3.12)"` and `poetry install`. '
        "See COLAB_RUNBOOK.md section 7."
    )
print(f"Poetry venv Python {venv_version} OK")

In [ ]:
!poetry run python -c "import torch; print(torch.cuda.is_available(), torch.cuda.device_count(), torch.cuda.get_device_name(0))"

## 4. Wire up the dataset and results/ (Drive)

`make colab-setup` runs `scripts/colab/setup_colab.sh`: it verifies Drive is
mounted, builds `daninhas_full.zip` on Drive on the **first session only**
(later sessions reuse it), unzips it onto the local runtime disk for fast
reads, and symlinks `results/` to Drive so every artifact survives a
disconnect. Idempotent -- safe to re-run after a disconnect. See
`COLAB_RUNBOOK.md` section 4.

In [ ]:
!make colab-setup

## 5. Sanity checks

See `COLAB_RUNBOOK.md` section 5.

In [ ]:
!make test

In [ ]:
!make smoke

In [ ]:
!make colab-check

### GPU identity check

Confirm this is the GPU you intended **before launching the campaign**. The cell below reads the
`run_metadata.json` written by the check run above (`environment` and `determinism` blocks).

In [ ]:
import glob
import json

hits = sorted(glob.glob("results/colab_check/**/run_metadata.json", recursive=True))
assert hits, "no run_metadata.json under results/colab_check/ -- did `make colab-check` finish?"
meta = json.load(open(hits[0]))
env = meta["environment"]
gpus = env.get("gpus") or []
assert gpus, "no GPU recorded in run_metadata.json -- is this a GPU runtime?"
gpu = gpus[0]
print("GPU name:      ", gpu.get("name"))
print("GPU memory GB: ", gpu.get("total_memory_gb"))
print("driver:        ", gpu.get("driver_version"))
print("CUDA (torch):  ", env["torch"].get("cuda"), "| cuDNN:", env["torch"].get("cudnn"))
print("determinism:   ", meta["determinism"])

In [ ]:
!poetry run python tools/loader.py --model results/colab_check/daninhas_full/SEED_1/NQ_100_NIL_100_NR_1_NE_10/SSRAEKmeansHCSampling/saved_model.pth

In [ ]:
import glob

img = sorted(glob.glob("DATA/daninhas_full/test/DATASET_GRAMINEA/*"))[0]
print(img)
!poetry run python tools/predict.py --model results/colab_check/daninhas_full/SEED_1/NQ_100_NIL_100_NR_1_NE_10/SSRAEKmeansHCSampling/saved_model.pth --image "{img}"

**Only continue to section 6 once all of the above pass.**

## 6. The campaign -- the single path for the final execution

One manifest drives the single, deduplicated execution of everything the three papers need
(paper 1: 12 classical strategies + KMH x nq {10,50,100} + the `FullSupervised` upper bound;
paper 2/3: TexHAL/RNHAL ablations at nq=100): **222 runs / 74 run groups**. See `COLAB_RUNBOOK.md`
"The campaign" and `.specs/experiments/campaign.md`. The `PART` parameter selects `all` or a subset
(one part per session, see below). Budget Drive space for ~222 checkpoints (~21 GB).

### 6b. KMH parameter sweep (ADR 0012)

The KMH parameter sweep is complete (executed 2026-11-07 to 2026-11-10). To revisit it, point the Parameters cell at the sweep manifest (paper 1; 165 runs / 55 groups, stage 2 included, results in the
NEW tree `results/campaign_kmh/`, about 13 GB of checkpoints on Drive) with `PART = "all"`: the run cell
executes the three parts in the order `kmh_nq100`, `kmh_nq50`, `kmh_nq10` and, after a disconnect, resumes
where it stopped once sections 0-4 are redone. Nothing needs editing between sessions. The cells that follow
(list, run, progress, verify, report, Drive copy) honour `MANIFEST` automatically; set `MANIFEST = ""` to go
back to the main campaign. Wall-clock is unmeasured: record the hours printed by the run cell in
`COLAB_RUNBOOK.md` section 6b.

| `PART` | Runs | Groups |
|---|---|---|
| `kmh_nq100` | 33 | 11 |
| `kmh_nq50` | 51 | 17 |
| `kmh_nq10` | 51 | 17 |

### 6c. Hierarchy transfer runs (ADR 0013)

Twelve runs (4 groups x seeds 1-3, `n_query=100`) of the KMH-sweep winner shape (`k=[600,100]`, no resampling) and of the
reference shape without resampling, with the SSRAE (RNHAL) and VCTex Q=17 (TexHAL) representations. They are a part of the
MAIN campaign manifest (executed 2026-11-11, verified). To revisit it set `PART = "hier_transfer"` in the Parameters cell, run
cells 0-4 and then the cells below (list, run, verify with `PART` restricting to the part, report). Results go to `results/campaign/rnhal/6_2/` and
`results/campaign/texhal_q17/6_2/`. Estimated 2 to 4 GPU hours. See `COLAB_RUNBOOK.md` section 6c.

### 6d. Re-run of the seven fixed paper-1 baselines (ADR 0014)

63 runs (21 groups x seeds 1-3) of `AdversarialBIM`, `KMeansSampling`, `KCenterGreedy`, `LeastConfidenceDropout`,
`MarginSamplingDropout`, `EntropySamplingDropout` and `BALDDropout` at `n_query` 10/50/100, after the KI-41/KI-42/KI-45 fixes,
split over **two Colab accounts at the same time**: account A runs `PART = "paper1_rerun_a"` (24 runs, about 4.3 GPU hours,
about 2.3 GB of checkpoints), account B runs `PART = "paper1_rerun_b"` (39 runs, about 4.2 GPU hours, about 3.7 GB). The new
runs land in the SAME leaf paths as the first execution (`results/campaign/paper1/nq*/...`), so the superseded leaves are
first MOVED (never deleted) to `results/old_results_broken/` by the three cells below: dry run, move, check. On a Drive
without the old campaign results the first two report `missing=63` and move nothing, which is fine; the check must say
`CHECK OK` before the run cell (the check cell and the gate cell right before the run RAISE on failure, so
"Run all" stops there; the engine itself also refuses to run while a superseded leaf is live). The gate cell also
asserts that `campaign-list` reports the expected count for `PART` (24 / 39). **On first use run cell by cell**, not
"Run all". **Account B must set `PART = "paper1_rerun_b"`** and see `39 runs (13 run groups)` in the Parameters
banner. Then: run, verify (expect `OK=24` / `OK=39`, `STALE=0`) and the zip cell after Verify. Full guide:
`COLAB_RUNBOOK.md` section 6d.

In [ ]:
!make campaign-archive-superseded DRY=1

In [ ]:
!make campaign-archive-superseded

In [ ]:
import subprocess

# Raises CalledProcessError while a superseded or unreadable leaf is still in the live tree, so "Run all" stops here.
subprocess.run(["make", "campaign-archive-superseded", "CHECK=1"], cwd=REPO_DIR, check=True)


In [ ]:
!make campaign-list {MANIFEST_ARG} PART={PART}

### Splitting the campaign across sessions

For the KMH sweep keep `PART = "all"` and simply re-run the run cell in every session. For the main
campaign (210 runs executed; `hier_transfer`, 12 runs, ADR 0013, prepared 2026-11-10) one `PART` per session was used (edit `PART` in the Parameters cell, re-run
cells 0-4, then the run cell):

| Session | `PART` | Runs |
|---|---|---|
| 1 | `paper1` | 117 |
| 2 | `rnhal` | 42 |
| 3 | `texhal` | 30 |
| 3b | `texhal_q17` | 18 |
| 3c | `hier_transfer` | 12 |
| 4 | `upper_bound` | 3 |
| 5 (ADR 0014, two accounts in parallel) | `paper1_rerun_a` / `paper1_rerun_b` | 24 / 39 (re-run of 63 of the 117 `paper1` runs) |

Every run is skipped if its results already exist, so re-running the same `PART` after a disconnect or
session limit **resumes** where it stopped (at most the run in progress is lost). Use the Progress cell
below at any time to see how far along a part is.

Run (skip-existing: **re-running this cell resumes after a disconnect** -- redo sections 0-4 first).
The GPU identity was already confirmed by the check cell in section 5.


In [ ]:
import re
import subprocess

# Pre-run gate (ADR 0014): every failure RAISES, so "Run all" stops here instead of launching a stale or wrong part.
if MANIFEST == "":
    subprocess.run(["make", "campaign-archive-superseded", "CHECK=1"], cwd=REPO_DIR, check=True)
list_cmd = ["make", "campaign-list", f"PART={PART}"] + ([MANIFEST_ARG] if MANIFEST_ARG else [])
listing = subprocess.run(list_cmd, cwd=REPO_DIR, check=True, capture_output=True, text=True).stdout
total = re.search(r"^\s*TOTAL\s+(\d+) runs \((\d+) run groups\)", listing, flags=re.M)
assert total, "campaign-list printed no TOTAL line:\n" + listing[-2000:]
print(total.group(0).strip())
if EXPECTED_RUNS is not None:
    assert (int(total.group(1)), int(total.group(2))) == (EXPECTED_RUNS, EXPECTED_GROUPS), (
        f"PART={PART}: campaign-list reports {total.group(1)} runs ({total.group(2)} run groups), "
        f"expected {EXPECTED_RUNS} ({EXPECTED_GROUPS})"
    )
print(f"Gate OK: PART={PART}, {total.group(1)} runs. The next cell launches them.")


In [ ]:
import time

start = time.time()
!make campaign-run {MANIFEST_ARG} PART={PART}

In [ ]:
elapsed_hours = (time.time() - start) / 3600
print(f"Campaign part(s) {PART} took {elapsed_hours:.2f} hours on this runtime.")
print("Paste this number into COLAB_RUNBOOK.md's campaign section (wall-clock TBD).")

**Progress** (usable anytime, e.g. from a second cell while the run is going, or after a disconnect):

In [ ]:
!make campaign-verify {MANIFEST_ARG} PART={PART} 2>&1 | tail -n 15
!tail -n 20 {RESULTS_ROOT}/campaign.log 2>/dev/null || echo "no campaign.log yet"

In [ ]:
!cat {RESULTS_ROOT}/failures.log 2>/dev/null || echo "no failures logged"

**Verify**: OK/INCOMPLETE/MISSING per job plus the seed-consistency audit (every run of a seed must start
from the identical initial labeled set). Exit code 0 = all OK, 1 = incomplete/missing job or FAIL audit,
2 = only audit WARNs. Re-run the run cell above to redo failed/missing jobs.

In [ ]:
!make campaign-verify {MANIFEST_ARG} PART={PART}

**6d only: zip the new leaves of the re-run part** for download. The cell refuses (raises) unless the superseded-leaf
check passes and Verify reports every run of the part `OK`; it adds this account's `campaign.log` (and `failures.log`)
as `results/campaign/campaign_{PART}.log` so the measured times survive the merge, and copies `{PART}.zip` to the root
of this account's Drive.

In [ ]:
import re
import shutil
import subprocess

# 6d only (ADR 0014): zip the NEW leaves of the re-run part, plus this account's campaign.log / failures.log
# under part-specific names (results/campaign/campaign_<PART>.log), so the measured times survive the merge.
# REFUSES (raises) unless the superseded-leaf check passes and verify reports all runs of the part OK.
ZIP_PARTIAL = False  # set True only to ship an incomplete part on purpose (e.g. no credits left); default refuses
RERUN_LEAF_GLOBS = {
    "paper1_rerun_a": [
        "results/campaign/paper1/nq100/*/SEED_*/NQ_100_*/AdversarialBIM",
        "results/campaign/paper1/nq50/*/SEED_*/NQ_50_*/AdversarialBIM",
        "results/campaign/paper1/nq*/*/SEED_*/NQ_*/KMeansSampling",
        "results/campaign/paper1/nq*/*/SEED_*/NQ_*/KCenterGreedy",
    ],
    "paper1_rerun_b": [
        "results/campaign/paper1/nq10/*/SEED_*/NQ_10_*/AdversarialBIM",
        "results/campaign/paper1/nq*/*/SEED_*/NQ_*/LeastConfidenceDropout",
        "results/campaign/paper1/nq*/*/SEED_*/NQ_*/MarginSamplingDropout",
        "results/campaign/paper1/nq*/*/SEED_*/NQ_*/EntropySamplingDropout",
        "results/campaign/paper1/nq*/*/SEED_*/NQ_*/BALDDropout",
    ],
}
if PART in RERUN_LEAF_GLOBS:
    subprocess.run(["make", "campaign-archive-superseded", "CHECK=1"], cwd=REPO_DIR, check=True)
    verify = subprocess.run(["make", "campaign-verify", f"PART={PART}"], cwd=REPO_DIR, capture_output=True, text=True)
    totals = re.search(r"Totals: OK=(\d+) .*", verify.stdout)
    print(totals.group(0) if totals else verify.stdout[-2000:] + verify.stderr[-2000:])
    n_ok = int(totals.group(1)) if totals else 0
    if n_ok != EXPECTED_RUNS and not ZIP_PARTIAL:
        raise RuntimeError(f"verify reports OK={n_ok}, expected {EXPECTED_RUNS}: re-run the run cell (resumes) first")
    zip_path = f"/content/{PART}.zip"
    subprocess.run(f"rm -f {zip_path} && zip -qr {zip_path} " + " ".join(RERUN_LEAF_GLOBS[PART]),
                   shell=True, cwd=REPO_DIR, check=True)
    stage = f"/content/zip_stage_{PART}"
    shutil.rmtree(stage, ignore_errors=True)
    os.makedirs(f"{stage}/{RESULTS_ROOT}")
    for name in ("campaign", "failures"):
        log = f"{REPO_DIR}/{RESULTS_ROOT}/{name}.log"
        if os.path.exists(log):
            shutil.copy(log, f"{stage}/{RESULTS_ROOT}/{name}_{PART}.log")
    subprocess.run(f"cd {stage} && zip -qr {zip_path} results", shell=True, check=True)
    listing = subprocess.run(["unzip", "-l", zip_path], capture_output=True, text=True, check=True).stdout
    n_models = listing.count("saved_model.pth")
    print(f"{zip_path}: {n_models} checkpoints;", "campaign log included" if f"campaign_{PART}.log" in listing else "NO campaign.log")
    if n_models != (n_ok if ZIP_PARTIAL else EXPECTED_RUNS):
        raise RuntimeError(f"zip holds {n_models} checkpoints, expected {n_ok if ZIP_PARTIAL else EXPECTED_RUNS}")
    subprocess.run(["ls", "-lh", zip_path], check=True)
    subprocess.run(["cp", zip_path, f"{DRIVE_ROOT}/"], check=True)
else:
    print(f"PART={PART} is not a re-run part (paper1_rerun_a / paper1_rerun_b): nothing to zip.")


**Report** + copy to Drive (tables md/tex/csv, `seed_audit.md`, mean confusion matrices).

In [ ]:
!make campaign-report {MANIFEST_ARG} {REPORT_ARGS}

In [ ]:
!mkdir -p "$DRIVE_ROOT/results"
!cp -r "$REPORT_DIR" "$DRIVE_ROOT/results/$DRIVE_TABLES"

In [ ]:
!ls "$DRIVE_ROOT/results/$DRIVE_TABLES"

## 7. Troubleshooting

See `COLAB_RUNBOOK.md` section 7 for full detail. Quick pointers:

- Drive FUSE slowness / `Transport endpoint is not connected` -> remount
  Drive (section 1) and re-run `!make colab-setup`.
- `results/` symlink or local `results/` conflicts -> read
  `scripts/colab/setup_colab.sh`'s error message; it names the exact
  conflicting path.
- Out of GPU memory -> copy the params JSON into gitignored `files_config/local/`
  and lower `batch_size` there; never edit the committed configs in place.
- Session limit reached -> re-run sections 0-4 in a fresh runtime, then the
  run cell again (skip-existing resumes; only complete leaves are skipped).
- Python 3.13 system runtime (expected on Colab) -> handled by the uv-managed 3.12
  in section 3; see `COLAB_RUNBOOK.md` section 7 if the venv check fails.